In [ ]:
import json
import numpy as np
np.set_printoptions(precision=2, suppress=True)
from nerfstudio.cameras import camera_utils
import torch
import os
import yaml
from pathlib import Path
from f3rm.manual.instance_axes_annotator import AxesAnnotator
import json
import cv2
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from f3rm.features.orientany.orientany_main import OrientAny
from f3rm.features.utils import (
    build_transform_lookup, 
    get_nerf_ccs_to_normal_ccs_T, 
    get_nerf_ccs_to_orig_nerf_world,
    get_orig_to_final_nerf_world_transform_scale
)

In [ ]:
DATASET_TRANSFORMS_PATH = "datasets/f3rm/panda_demos/caterpillar/transforms.json"
T_orig_to_final_nerf_world, orig_to_final_nerf_world_scale = get_orig_to_final_nerf_world_transform_scale(DATASET_TRANSFORMS_PATH)
dataset_transforms_data = json.load(open(DATASET_TRANSFORMS_PATH, "r"))
transforms_lookup = build_transform_lookup(dataset_transforms_data["frames"])

In [ ]:
# initial image (=tt2.png)
image_path1 = "datasets/f3rm/panda_demos/caterpillar/images/frame_00013.jpg"
pil_img1 = Image.open(image_path1).convert('RGB')
image_path2 = "datasets/f3rm/panda_demos/caterpillar/images/frame_00050.jpg"
pil_img2 = Image.open(image_path2).convert('RGB')

In [ ]:
nerf_ccs1_to_orig_nerf_world = get_nerf_ccs_to_orig_nerf_world(os.path.basename(image_path1), transforms_lookup)
nerf_ccs2_to_orig_nerf_world = get_nerf_ccs_to_orig_nerf_world(os.path.basename(image_path2), transforms_lookup)

In [ ]:
nerf_ccs1_to_final_nerf_world = T_orig_to_final_nerf_world @ nerf_ccs1_to_orig_nerf_world
nerf_ccs1_to_final_nerf_world[:3,3] *= orig_to_final_nerf_world_scale
nerf_ccs2_to_final_nerf_world = T_orig_to_final_nerf_world @ nerf_ccs2_to_orig_nerf_world
nerf_ccs2_to_final_nerf_world[:3,3] *= orig_to_final_nerf_world_scale

In [ ]:
nerf_ccs1_to_nerf_ccs2 = np.linalg.inv(nerf_ccs2_to_final_nerf_world) @ nerf_ccs1_to_final_nerf_world

In [ ]:
normal_ccs1_to_final_nerf_world = nerf_ccs1_to_final_nerf_world @ np.linalg.inv(get_nerf_ccs_to_normal_ccs_T())
normal_ccs2_to_final_nerf_world = nerf_ccs2_to_final_nerf_world @ np.linalg.inv(get_nerf_ccs_to_normal_ccs_T())
normal_ccs1_to_normal_ccs2 = np.linalg.inv(normal_ccs2_to_final_nerf_world) @ normal_ccs1_to_final_nerf_world

In [ ]:
orient_any = OrientAny("f3rm/features/orientany/ckpts", "ronormsigma1_dino_weight.pt")

In [ ]:
def get_orientany_R_objw_to_normal_ccs(pil_img, oany_obj: OrientAny, do_remove_background=True):
    rm_bkg_img = orient_any.preprocess_remove_bkg(pil_img, do_remove_background=do_remove_background)
    outs = oany_obj.get_model_outputs(rm_bkg_img, viz_distn=False)
    R_objw_to_normal_ccs = orient_any.get_R_objw2cam(outs['phi'], outs['theta_elev'], outs['delta'])
    return R_objw_to_normal_ccs

In [ ]:
R_objw_to_normal_ccs1 = get_orientany_R_objw_to_normal_ccs(pil_img1, orient_any)
R_objw_to_normal_ccs2 = get_orientany_R_objw_to_normal_ccs(pil_img2, orient_any)
R_objw_to_nerf_ccs1 = get_nerf_ccs_to_normal_ccs_T()[:3,:3].T @ R_objw_to_normal_ccs1
R_objw_to_nerf_ccs2 = get_nerf_ccs_to_normal_ccs_T()[:3,:3].T @ R_objw_to_normal_ccs2

In [ ]:
axes_image1 = AxesAnnotator.visualize_rotation_matrix(cv2.cvtColor(np.array(pil_img1), cv2.COLOR_RGB2BGR), (pil_img1.width//2, pil_img1.height//2), R_objw_to_nerf_ccs1)
plt.imshow(cv2.cvtColor(axes_image1, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
axes_image2 = AxesAnnotator.visualize_rotation_matrix(cv2.cvtColor(np.array(pil_img2), cv2.COLOR_RGB2BGR), (pil_img2.width//2, pil_img2.height//2), R_objw_to_nerf_ccs2)
plt.imshow(cv2.cvtColor(axes_image2, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
predR_objw_to_nerf_ccs2 = nerf_ccs1_to_nerf_ccs2[:3,:3] @ R_objw_to_nerf_ccs1

In [ ]:
pred_axes_image2 = AxesAnnotator.visualize_rotation_matrix(cv2.cvtColor(np.array(pil_img2), cv2.COLOR_RGB2BGR), (pil_img2.width//2, pil_img2.height//2), predR_objw_to_nerf_ccs2)
plt.imshow(cv2.cvtColor(pred_axes_image2, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
predR_objw_to_normal_ccs2 = get_nerf_ccs_to_normal_ccs_T()[:3,:3] @ predR_objw_to_nerf_ccs2

In [ ]:
pred_outs_cam1_to_2 = orient_any._angles_from_R(predR_objw_to_normal_ccs2)
print(pred_outs_cam1_to_2)

In [ ]:
# Distribution propagation approach: pred on cam1 -> propagate distn to cam2 -> argmax -> print
# Get full distributions from cam1
rm_bkg_img1 = orient_any.preprocess_remove_bkg(pil_img1, do_remove_background=True)
outs1 = orient_any.get_model_outputs(rm_bkg_img1, viz_distn=True)

In [ ]:
# Convert logits to probabilities
probs_phi = F.softmax(torch.from_numpy(outs1['gaus_ax_logits']), dim=0)
probs_theta = F.softmax(torch.from_numpy(outs1['gaus_pl_logits']), dim=0)
probs_delta = F.softmax(torch.from_numpy(outs1['gaus_ro_logits']), dim=0)

In [ ]:
# Propagate distributions to cam2
# Note: nerf_ccs1_to_nerf_ccs2 is the rotation from cam1 to cam2
probs_phi_cam2, probs_theta_cam2, probs_delta_cam2 = orient_any.push_distributions_to_new_view(
    probs_phi, probs_theta, probs_delta, 
    torch.from_numpy(normal_ccs1_to_normal_ccs2[:3, :3].astype(np.float32))
)

In [ ]:
# Get argmax from propagated distributions
phi_argmax_cam2 = torch.argmax(probs_phi_cam2).item()
theta_argmax_cam2 = torch.argmax(probs_theta_cam2).item() - 90  # Convert back to theta_elev range
delta_argmax_cam2 = torch.argmax(probs_delta_cam2).item() - orient_any.model_config['ro_offset']

In [ ]:
print("Distribution propagation approach:")
print(f"φ={phi_argmax_cam2:.1f}°, θ={theta_argmax_cam2:.1f}°, δ={delta_argmax_cam2:.1f}°")
print()
print("Original argmax propagation approach:")
print(f"φ={pred_outs_cam1_to_2['phi']:.1f}°, θ={pred_outs_cam1_to_2['theta_elev']:.1f}°, δ={pred_outs_cam1_to_2['delta']:.1f}°")